# LAMBADA Evaluation for DualStreamLLM
This notebook evaluates the DualStreamLLM using a separator model for deconvolution on the LAMBADA dataset.


In [1]:
model_name = 'Qwen/Qwen2.5-3B'
separator_model_name = 'Qwen/Qwen2.5-0.5B'
checkpoint_path = None#'checkpoints/run_20260203_155520/checkpoint-20000/model.pt'

In [ ]:
import json
import torch
import random
import os 
import sys

import numpy as np
import matplotlib.pyplot as plt
import re
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F

# Add project root to path
sys.path.append(os.getcwd())

from generate_dual_stream import (
    load_checkpoint,
    DualLinearStrategy,
    EnsembleStrategy,
    SingleModelStrategy
)
from model import DualStreamLLM

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

ImportError: cannot import name 'DualLinearStrategy' from 'generate_dual_stream' (/mnt/extremessd10tb/tikhonov/projects/mixture-of-embeddings/generate_dual_stream.py)

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


if checkpoint_path:
    big_model = load_checkpoint(checkpoint_path, model_name, device)
    print("Checkpoint of finetuned DualStreamLLM loaded successfully")
else:
    big_model = DualStreamLLM(
        model_name=model_name,
        freeze_backbone=True,
        optimize_memory=False,
        embedding_init_strategy='copy',
        objective='distillation',
        shared_embeddings=True
    ).to(device)
    big_model.eval()
    print("Pretrained DualStreamLLM loaded successfully")


# Load Separator Model
print(f"Loading separator model: {separator_model_name}")
separator_model = AutoModelForCausalLM.from_pretrained(
    separator_model_name,
    torch_dtype=torch.bfloat16
).to(device)
separator_model.eval()


print("Models loaded successfully.")

In [ ]:
def run_lambada_evaluation(
    tokenizer=None,
    big_model=None,
    dataset_path=None,
    device=None,
    strategy=None,
    temperature=None,
    num_examples_to_print=None,
    max_new_tokens=None,
    model_name=None,
    separator_model_name=None,
    checkpoint_path=None,
):
    # Validate strategy
    if strategy is None:
        raise ValueError("strategy parameter is required. Pass DualLinearStrategy or EnsembleStrategy instance.")
    
    print(f"Starting LAMBADA evaluation with configuration:")
    print(f"  Model: {model_name}")
    print(f"  Separator: {separator_model_name}")
    print(f"  Strategy: {strategy.get_config_string()}")
    print(f"  Device: {device}")
    print(f"  Temperature: {temperature}")
    print(f"  Max tokens: {max_new_tokens}\n")
    
    # Load and sort dataset
    with open(dataset_path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    print("Sorting dataset by length to optimize pairing...")
    data_sorted = sorted(data, key=lambda x: len(tokenizer.encode(x['prompt'])))
    
    # Pair examples with similar lengths
    pairs = []
    for i in range(0, len(data_sorted) - 1, 2):
        pairs.append((data_sorted[i], data_sorted[i+1]))
    
    # Shuffle pairs
    random.seed(42)
    random.shuffle(pairs)
    
    correct_a = 0
    correct_b = 0
    total = 0
    printed_count = 0
    results_a = []
    results_b = []
    
    pbar = tqdm(pairs, desc="Evaluating LAMBADA")
    
    for item_a, item_b in pbar:
        # Tokenize prompts
        prompt_tokens_a = tokenizer(item_a['prompt'], return_tensors='pt')['input_ids'].to(device)
        prompt_tokens_b = tokenizer(item_b['prompt'], return_tensors='pt')['input_ids'].to(device)
        
        # Left-truncate to match lengths
        len_a = prompt_tokens_a.shape[1]
        len_b = prompt_tokens_b.shape[1]
        min_len = min(len_a, len_b)
        
        input_tokens_a = prompt_tokens_a[:, -min_len:]
        input_tokens_b = prompt_tokens_b[:, -min_len:]
        
        # Inference using selected strategy
        with torch.no_grad():
            out_a, out_b = strategy.generate(
                big_model,
                input_tokens_a,
                input_tokens_b,
                tokenizer,
                max_new_tokens=max_new_tokens,
                temperature=temperature
            )
        
        # Check correctness
        full_correct_text_a = item_a['prompt'] + " " + item_a['completion']
        full_correct_text_b = item_b['prompt'] + " " + item_b['completion']
        
        generated_text_a = tokenizer.decode(out_a[0, :min_len + 1], skip_special_tokens=True)
        generated_text_b = tokenizer.decode(out_b[0, :min_len + 1], skip_special_tokens=True)
        
        is_correct_a = full_correct_text_a.startswith(generated_text_a)
        is_correct_b = full_correct_text_b.startswith(generated_text_b)
        
        correct_a += int(is_correct_a)
        correct_b += int(is_correct_b)
        total += 1
        
        results_a.append(is_correct_a)
        results_b.append(is_correct_b)
        
        # Print examples
        if printed_count < num_examples_to_print:
            print(f"\n--- Example Pair {printed_count + 1} [Length: {min_len} tokens] ---")
            print(f"Stream A Prompt: {item_a['prompt']}")
            print(f"Stream A Target Completion: '{item_a['completion']}'")
            
            first_gen_token_id_a = out_a[0, min_len].item()
            first_gen_token_a = tokenizer.convert_ids_to_tokens(first_gen_token_id_a)
            print(f"Stream A First Generated Token: '{first_gen_token_a}' (ID: {first_gen_token_id_a})")
            
            if max_new_tokens > 1:
                full_gen_a = tokenizer.decode(out_a[0, min_len:], skip_special_tokens=True)
                print(f"Stream A Full Generation: '{full_gen_a}'")
            
            print(f"Stream A Result: {'✓ CORRECT' if is_correct_a else '✗ WRONG'}")
            print("-" * 15)
            
            print(f"Stream B Prompt: {item_b['prompt']}")
            print(f"Stream B Target Completion: '{item_b['completion']}'")
            
            first_gen_token_id_b = out_b[0, min_len].item()
            first_gen_token_b = tokenizer.convert_ids_to_tokens(first_gen_token_id_b)
            print(f"Stream B First Generated Token: '{first_gen_token_b}' (ID: {first_gen_token_id_b})")
            
            if max_new_tokens > 1:
                full_gen_b = tokenizer.decode(out_b[0, min_len:], skip_special_tokens=True)
                print(f"Stream B Full Generation: '{full_gen_b}'")
            
            print(f"Stream B Result: {'✓ CORRECT' if is_correct_b else '✗ WRONG'}")
            printed_count += 1
            print()
            print('=' * 70)
        
        pbar.set_postfix({
            "Acc A": f"{correct_a/total:.4f}", 
            "Acc B": f"{correct_b/total:.4f}",
            "Acc Total": f"{(correct_a+correct_b)/(2*total):.4f}"
        })
    
    # Print results summary
    print(f"\nFinal Evaluation Results:")
    print(f"Total pairs processed: {total}")
    print(f"Accuracy Stream A: {correct_a/total:.4f}")
    print(f"Accuracy Stream B: {correct_b/total:.4f}")
    print(f"Total Accuracy: {(correct_a + correct_b)/(2 * total):.4f}")
    
    # Bootstrap analysis
    print("\nPerforming bootstrap analysis...")
    n_bootstrap = 1000
    np.random.seed(42)
    
    bootstrap_acc_a = []
    bootstrap_acc_b = []
    bootstrap_acc_total = []
    
    for _ in range(n_bootstrap):
        indices = np.random.choice(len(results_a), size=len(results_a), replace=True)
        acc_a = np.mean([results_a[i] for i in indices])
        acc_b = np.mean([results_b[i] for i in indices])
        acc_total = (acc_a + acc_b) / 2
        
        bootstrap_acc_a.append(acc_a)
        bootstrap_acc_b.append(acc_b)
        bootstrap_acc_total.append(acc_total)
    
    # Calculate confidence intervals
    ci_95_a = np.percentile(bootstrap_acc_a, [2.5, 97.5])
    ci_95_b = np.percentile(bootstrap_acc_b, [2.5, 97.5])
    ci_95_total = np.percentile(bootstrap_acc_total, [2.5, 97.5])
    
    # Store bootstrap distributions and confidence intervals for later use
    print("Bootstrap analysis complete.")

    

    
    # Return results dictionary
    return {
        'correct_a': correct_a,
        'correct_b': correct_b,
        'total': total,
        'acc_a': correct_a / total,
        'acc_b': correct_b / total,
        'acc_total': (correct_a + correct_b) / (2 * total),
        'ci_95_a': ci_95_a,
        'ci_95_b': ci_95_b,
        'ci_95_total': ci_95_total,
        'bootstrap_acc_total': bootstrap_acc_total,
        'model_name': model_name,
        'separator_model_name': separator_model_name,
        'strategy_config': strategy.get_config_string(),
    }


## Experiment 1: DualStreamLLM (Qwen2.5-3B) with Separator (Qwen2.5-0.5B)

In [ ]:
results_exp1 = None

# Ensemble strategy with DualStreamLLM and small separator model
strategy = EnsembleStrategy(separator_model=separator_model, gamma=1.0)
results_exp1 = run_lambada_evaluation(
    tokenizer=tokenizer,
    big_model=big_model,
    dataset_path='lambada_test_filtered.jsonl',
    device=device,
    strategy=strategy,
    temperature=0.0001,
    num_examples_to_print=0,
    max_new_tokens=1,
    model_name=model_name,
    separator_model_name=separator_model_name,
    checkpoint_path=checkpoint_path
)

## Experiment 2: Single Qwen2.5-3B (Baseline)

In [ ]:
results_exp2 = None

# Single model strategy for big model
strategy = SingleModelStrategy(model=big_model)
results_exp2 = run_lambada_evaluation(
    tokenizer=tokenizer,
    big_model=big_model,
    dataset_path='lambada_test_filtered.jsonl',
    device=device,
    strategy=strategy,
    temperature=0.0001,
    num_examples_to_print=0,
    max_new_tokens=1,
    model_name=model_name,
    separator_model_name=None,
    checkpoint_path=checkpoint_path
)

## Experiment 3: Single Qwen2.5-0.5B (Small Model Baseline)

In [ ]:
results_exp3 = None

# Single model strategy for small separator model
strategy = SingleModelStrategy(model=separator_model)
results_exp3 = run_lambada_evaluation(
    tokenizer=tokenizer,
    big_model=separator_model,
    dataset_path='lambada_test_filtered.jsonl',
    device=device,
    strategy=strategy,
    temperature=0.0001,
    num_examples_to_print=0,
    max_new_tokens=1,
    model_name=separator_model_name,
    separator_model_name=None,
    checkpoint_path=None,
)

## Summary of Results

In [ ]:
from tabulate import tabulate

# Prepare summary table
experiments = [
    {
        'name': 'DualStreamLLM + Separator (0.5B)',
        'model': 'Qwen2.5-3B',
        'separator': 'Qwen2.5-0.5B',
        'results': results_exp1
    },
    {
        'name': 'Single Qwen2.5-3B',
        'model': 'Qwen2.5-3B',
        'separator': 'None',
        'results': results_exp2
    },
    {
        'name': 'Single Qwen2.5-0.5B',
        'model': 'Qwen2.5-0.5B',
        'separator': 'None',
        'results': results_exp3
    }
]

# Build table data
table_data = []
for exp in experiments:
    if exp['results'] is not None:
        table_data.append([
            exp['name'],
            f"{exp['results']['acc_total']:.4f}",
            f"[{exp['results']['ci_95_total'][0]:.4f}, {exp['results']['ci_95_total'][1]:.4f}]"
        ])

# Print table
headers = ['Experiment', 'Accuracy', '95% CI']
print("\n" + "="*70)
print("LAMBADA Evaluation Results")
print("="*70)
print(tabulate(table_data, headers=headers, tablefmt='grid'))
print("="*70 + "\n")

# Create combined visualization
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

colors = ['#2ecc71', '#3498db', '#e74c3c']
exp_names = [exp['name'] for exp in experiments]

# Plot histograms for each experiment
for i, exp in enumerate(experiments):
    if exp['results'] is not None:
        ax.hist(exp['results']['bootstrap_acc_total'], 
                bins=30, 
                color=colors[i], 
                alpha=0.5, 
                label=f"{exp['name']} (μ={exp['results']['acc_total']:.4f})",
                edgecolor='black')

ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('LAMBADA Bootstrap Accuracy Distribution - All Experiments', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('lambada_summary_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Summary visualization saved as 'lambada_summary_results.png'")